# KNN Implementation, Evaluation & Reflection

**K-Nearest Neighbors Workshop · CSCN8010 · Final_Project_Group 1**
**Section owner:** Antonio Sainz — rubric criteria 3 (*KNN Implementation & Model Evaluation*, 3 pts) and 5 (*Reflection, Interpretation & Peer Comparison*, 2 pts)

**Goal:** predict the hourly weather condition in Waterloo, ON — **Clear, Cloudy, Rain or Snow** — from measured conditions (temperature, humidity, pressure, wind, cloud cover, hour, month), using scikit-learn's `KNeighborsClassifier`.

This notebook covers the last two steps of the team pipeline:

`DataLoader` (John) → `Preprocessor` (Eche) → **`KNNModel` → `Evaluator`** (Antonio)

The two classes live in [`src/knn_model.py`](src/knn_model.py) so they can be imported into the final `KNN_Workshop_Solution.ipynb` without copying code.

## 0. Setup

Locate the project root, make `src/` importable, and load the database settings from `.env`. The password never appears in the notebook.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().resolve()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")

from src.knn_model import KNNModel, Evaluator, RANDOM_STATE

LABELS = ["Clear", "Cloudy", "Rain", "Snow"]
print("Project root:", project_root)

## 1. Warm-up: the same classes on the Iris dataset

Before using our weather data, we check that `KNNModel` and `Evaluator` behave correctly on the classic Iris dataset from Part 1 of the workshop. Iris is small, clean and well separated, so near-perfect scores are expected. If the classes failed here, the problem would be in our code, not in the data.

In [ ]:
iris = load_iris()
iris_model = KNNModel()

X_tr, X_te, y_tr, y_te = iris_model.split(iris.data, iris.target)
iris_model.tune(X_tr, y_tr)
iris_model.fit(X_tr, y_tr)

iris_metrics = Evaluator.metrics(y_te, iris_model.predict(X_te))
print(f"Best k = {iris_model.best_k} ({iris_model.best_metric})")
pd.Series(iris_metrics).round(3)

**What this means:** the pipeline classifies every Iris test flower correctly with a small k. This confirms the split, tuning and scoring logic works, so we can move on to the real-world dataset with confidence.

## 2. Input data

> **Integration note:** in the final notebook this cell is replaced by John's `DataLoader` and Eche's `Preprocessor`. The stand-in below follows the team spec exactly — same labels, same leakage removal, same engineered features — so my results carry over once the real classes are connected.

The data is read from the team's Neon table `knn_weather` when it is available. Otherwise it falls back to the Open-Meteo CSV already in the repo, which holds the same 2025 hourly Waterloo data.

In [ ]:
def load_weather():
    url = os.getenv("DATABASE_URL")
    if url:
        try:
            import psycopg2
            with psycopg2.connect(url, connect_timeout=10) as conn:
                df = pd.read_sql("SELECT * FROM knn_weather ORDER BY time", conn)
            if len(df):
                print(f"Loaded {len(df):,} rows from Neon table knn_weather")
                return df
        except Exception as exc:
            print("Neon not available, using the CSV instead:", type(exc).__name__)
    df = pd.read_csv(project_root / "data" / "waterloo_weather_2025.csv")
    print(f"Loaded {len(df):,} rows from data/waterloo_weather_2025.csv")
    return df


weather = load_weather()
weather["time"] = pd.to_datetime(weather["time"])

In [ ]:
# --- Temporary stand-in for Eche's Preprocessor (labels + features) ---
def to_condition(code):
    code = int(code)
    if code <= 1:
        return "Clear"
    if code <= 3:
        return "Cloudy"
    if 51 <= code <= 67 or 80 <= code <= 82:
        return "Rain"
    if 71 <= code <= 77 or 85 <= code <= 86:
        return "Snow"
    return np.nan


weather["condition"] = weather["weather_code"].map(to_condition)
weather["hour"] = weather["time"].dt.hour
weather["month"] = weather["time"].dt.month
weather = weather.dropna(subset=["condition"])

# weather_code, rain, snowfall and precipitation are excluded: they give away the label (leakage).
FEATURES = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
    "surface_pressure", "cloud_cover", "wind_speed_10m", "wind_direction_10m",
    "wind_gusts_10m", "hour", "month",
]
X = weather[FEATURES]
y = weather["condition"]

print("Missing values in features:", int(X.isna().sum().sum()))
(y.value_counts(normalize=True) * 100).round(1).rename("% of hours")

**What this means:** the classes are **imbalanced**. Cloudy and Clear make up about 80% of the hours, while Rain and Snow are only about 10% each. A model that always guessed "Cloudy" would already be right 45% of the time. That's why we stratify the split and judge the model mainly by **macro F1**, which gives Rain and Snow the same weight as the common classes.

## 3. Train / test split and scaling

- **80/20 split with `stratify=y`**, so the rare Rain and Snow hours appear in the same proportion in both sets.
- **`StandardScaler` fitted on the training set only**, then applied to the test set. KNN works on distances, so without scaling, pressure (around 970 hPa) would dominate humidity (0–100%) and hour (0–23).

In [ ]:
model = KNNModel()
X_train, X_test, y_train, y_test = model.split(X, y, test_size=0.2)

scaler = StandardScaler().fit(X_train)      # fit on train only -> no leakage
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train: {len(X_train):,} hours | Test: {len(X_test):,} hours")

## 4. Choosing k (and the distance metric)

For every k from 3 to 25, and for both Euclidean and Manhattan distance, we run **5-fold stratified cross-validation on the training set only**, scored with macro F1. The test set is not touched during tuning, so the final score is an honest estimate.

In [ ]:
tuning = model.tune(X_train_s, y_train)
print(f"Best: k = {model.best_k}, distance = {model.best_metric}")

fig = Evaluator.plot_k_tuning(tuning, model.best_k, model.best_metric)
fig.show()

tuning.sort_values("cv_mean", ascending=False).head(5).round(3)

**What this means:** macro F1 is highest at **k = 3** (about 0.77) and falls steadily to about 0.67 at k = 25. Larger k averages over more neighbours, which washes out the small Rain and Snow classes in favour of Cloudy. Euclidean and Manhattan are almost identical, so the choice of distance matters far less than the choice of k. *Section 7 explains why such a small k wins on this particular data.*

## 5. Final model on the test set

We train on the full training set with the tuned settings and predict the 1,752 held-out hours.

In [ ]:
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

evaluator = Evaluator(labels=LABELS)
comparison = evaluator.compare({
    "Majority baseline": evaluator.baseline(X_train_s, y_train, X_test_s, y_test),
    f"KNN (k={model.best_k})": evaluator.metrics(y_test, y_pred),
})
display(comparison)
evaluator.plot_metric_comparison(comparison).show()

**What this means:** KNN reaches about **87% accuracy and 0.80 macro F1**, against 45% accuracy and 0.16 macro F1 for always predicting "Cloudy". Macro recall (about 0.79) is lower than accuracy because the model is better at the common classes than at the rare ones. The per-class table below shows exactly where.

In [ ]:
evaluator.report(y_test, y_pred)

**What this means:** **Clear** (F1 about 0.95) and **Cloudy** (about 0.87) are predicted very well. **Rain** (about 0.67) and **Snow** (about 0.72) are harder: with fewer than 1,000 examples each, a new hour often has more Cloudy neighbours than Rain or Snow neighbours. Rain recall is the weakest number, at about 62%, so roughly 4 in 10 rainy hours are missed.

In [ ]:
evaluator.plot_confusion(y_test, y_pred).show()
evaluator.confusion(y_test, y_pred)

**What this means:** almost every mistake involves **Cloudy**. Rainy and snowy hours are mostly mislabelled as Cloudy (around 57 and 51 hours), which makes physical sense: an overcast hour just before or after precipitation has nearly the same temperature, humidity and pressure. Rain and Snow are almost never confused with each other, because temperature separates them cleanly.

## 6. How much does `cloud_cover` carry the model?

Open-Meteo derives weather codes 0–3 (Clear vs Cloudy) largely from cloud cover. To check how dependent the model is on that one feature, we retrain without it.

In [ ]:
no_cloud = [f for f in FEATURES if f != "cloud_cover"]
ablation = KNNModel()
Xa_tr, Xa_te, ya_tr, ya_te = ablation.split(X[no_cloud], y)
sc_a = StandardScaler().fit(Xa_tr)
ablation.tune(sc_a.transform(Xa_tr), ya_tr)
ablation.fit(sc_a.transform(Xa_tr), ya_tr)
ya_pred = ablation.predict(sc_a.transform(Xa_te))

evaluator.compare({
    "All features": evaluator.metrics(y_test, y_pred),
    "Without cloud_cover": evaluator.metrics(ya_te, ya_pred),
})

**What this means:** without `cloud_cover`, accuracy drops from about 0.87 to 0.75, and almost all of the loss comes from **Clear vs Cloudy**. Rain and Snow F1 barely change. So `cloud_cover` is the feature that separates sunny from overcast hours. It's a legitimate measured variable, not leakage, but the team should state it clearly when explaining the results.

## 7. Honesty check: are neighbouring hours leaking between train and test?

Weather changes slowly, so 2 PM and 3 PM on the same day are almost identical rows. A random split puts many of these "twin" hours on opposite sides, and KNN can then find a near-copy of each test hour in the training set. That would also explain why the smallest k (3) wins.

To test this, we cross-validate while keeping **whole weeks together**, so a week is either entirely in training or entirely in testing.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

weeks = weather["time"].dt.isocalendar().week
rows = []
for k in [3, 7, 15, 25]:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    random_cv = cross_val_score(pipe, X, y, scoring="f1_macro",
                                cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))
    week_cv = cross_val_score(pipe, X, y, groups=weeks, scoring="f1_macro",
                              cv=StratifiedGroupKFold(5, shuffle=True, random_state=RANDOM_STATE))
    rows.append({"k": k, "random split": random_cv.mean(), "weeks kept together": week_cv.mean()})

pd.DataFrame(rows).set_index("k").round(3)

**What this means:** when whole weeks are held out, macro F1 drops from about **0.80 to about 0.62**, and k = 3 no longer wins (k between 7 and 15 is just as good). Part of the headline score therefore comes from near-duplicate neighbouring hours. **About 0.62 macro F1 is the realistic estimate for predicting a week the model has never seen.** It's still roughly four times the baseline, but it's the number we should trust. We keep the 80/20 stratified split because the team spec requires it, and report both.

## 8. Reflection

### 1. How did KNN perform on our dataset?
On the required stratified 80/20 split, KNN (k = 3) reached about **87% accuracy and 0.80 macro F1**, compared with 45% and 0.16 for a majority-class baseline. Clear and Cloudy were predicted very well (F1 0.87–0.95), while Rain and Snow were harder (0.67–0.72), mostly being confused with Cloudy. When we held out whole weeks to remove near-duplicate hours, macro F1 fell to about **0.62**, which is the more realistic performance on genuinely new days.

### 2. Which preprocessing decisions affected performance most?
- **Removing leaking columns** (`weather_code`, `rain`, `snowfall`, `precipitation`): keeping them would have produced a near-perfect but meaningless model.
- **Scaling with `StandardScaler`**, fitted on training data only: KNN is distance-based, so unscaled pressure values would dominate every other feature.
- **Keeping `cloud_cover`**: removing it cut accuracy from about 0.87 to 0.75, almost entirely in Clear vs Cloudy.
- **Stratifying the split and using macro F1**, so the rare Rain and Snow classes are judged fairly.
- *Eche (preprocessing): add 2–3 points here.*

### 3. Strengths and weaknesses of KNN
**Strengths:** simple and intuitive; no training phase; naturally handles four classes; easy to explain ("this hour looks like these 3 past hours").
**Weaknesses:** sensitive to feature scaling and to irrelevant features; slow at prediction time, because every query is compared with all ~7,000 training hours; biased toward the majority class as k grows; and, as Section 7 showed, easily flattered by near-duplicate rows in time-series data.

### 4. When would we not use KNN in practice?
- Large or streaming datasets that need fast predictions, such as millions of rows or real-time systems.
- High-dimensional data, where distances lose meaning (the curse of dimensionality).
- Strongly imbalanced problems where missing the rare class is costly, such as a severe-weather warning.
- Time-dependent forecasting, where models that use time order (lag features, gradient boosting, sequence models) are a better fit.

### 5. How could this workflow be extended in an MLOps context?
- **Automation:** schedule the Open-Meteo pull into Neon (already scripted in `src/weather_stream.py`) and retrain on a fixed cadence.
- **CI/CD:** run the pipeline classes in GitHub Actions on every pull request, with tests that fail if macro F1 drops below a threshold.
- **Experiment tracking:** log k, distance metric, features and metrics with a tool such as MLflow.
- **Validation:** use time-based (week-grouped) validation as the default, given Section 7.
- **Monitoring:** compare live predictions with observed conditions and alert on drift, for example a shift between seasons.

### Peer comparison (other team)
*To be completed after our session with the other team:* their dataset, preprocessing choices, chosen k and macro F1, how they compare with ours, and what we learned from them.

### Team contributions
*Each teammate adds 2–3 points from their part.*
- **Sultan (pipeline):** _pending_
- **John (data acquisition):** _pending_
- **Eche (preprocessing):** _pending_
- **Antonio (model & evaluation):** small k wins mainly because of near-duplicate hours; week-grouped validation gives the honest number. `cloud_cover` drives Clear vs Cloudy. Rain recall (about 62%) is the main weakness to improve.